In [10]:
import numpy as np
import pandas as pd
from collections import Counter
import requests
from bs4 import BeautifulSoup

In [2]:
df_PT = pd.read_csv('published_texts.csv')

In [3]:
df_PT = df_PT[df_PT['cdli_id'].apply(lambda x: len(str(x)) == 7)].reset_index(drop = True)

In [4]:
unique_p_ids = list(set(df_PT['cdli_id']))

In [5]:
from bs4 import BeautifulSoup
import requests

def scrape_translations_and_transliterations(p_id):
    req_id = f'p{p_id[1:4]}'
    link = f'https://aicuneiform.com/p/{req_id}.json'
    try:
        response = requests.get(link)
        if response.status_code != 200:
            return None
        try:
            resp_json = response.json()
        except Exception:
            return None
        try:
            html = resp_json[p_id]['html']
        except KeyError:
            return None
    except Exception:
        return None

    soup = BeautifulSoup(html, 'html.parser')

    transliterations = soup.find_all('div', class_='lang-akk text')
    translations = soup.find_all('div', class_='lang-ml_en text')

    results = []
    for tlit, trans in zip(transliterations, translations):
        tlit_line = tlit.find('span', class_='line line-0')
        trans_line = trans.find('span', class_='line line-0')

        if tlit_line and trans_line:
            results.append((
                tlit_line.get_text(strip=True),
                trans_line.get_text(strip=True)
            ))

    return results

p_id_to_tuples = {}
failed_p_ids = []

for i,p_id in zip(range(len(unique_p_ids)) , unique_p_ids):
    if i%500 == 0:
        print(f'Done till {i}')
    tuples = scrape_translations_and_transliterations(p_id)
    if tuples is None:
        failed_p_ids.append(p_id)
    else:
        p_id_to_tuples[p_id] = tuples

Done till 0
Done till 500
Done till 1000
Done till 1500
Done till 2000
Done till 2500
Done till 3000
Done till 3500
Done till 4000
Done till 4500
Done till 5000
Done till 5500
Done till 6000
Done till 6500
Done till 7000


In [6]:
len(list(p_id_to_tuples.keys()))

4419

In [9]:
import json

with open('AICC_Extracted_Transliterations_And_Translations.json', 'w', encoding='utf-8') as f:
    json.dump(p_id_to_tuples, f, indent=4, ensure_ascii=False)

In [8]:
p_id_to_tuples

{'P361718': [('1(disz) 5/6(disz) _ma-na_ 1(disz) 5/6(disz) _gin2_',
   '1 5/6 mina 1 5/6 shekels'),
  ('isz-pu-ru-nim-ma a-na ta-aq-na-bi4-im u3 ma-nu-ki-a-szur i-na ka3-ni-isz ta#-di2-in _igi#_ a-szur-i-mi3-ti2 _dumu_ i-di2-we-er',
   "sent, and to Taqnabum and Manuki-Assur in the ka'nish he gave. Before Ashur-imitti, son of Idi-wer.")],
 'P358927': [('[...] x x x [...] [...]-ri#?-a li-x-[...] [...] x su2-en6 [...] sza# ta-asz2-pu-ra#-[ni] [...]-x-ti2-ka3 / szu-mi a-[x] [...] i-ku-pi3-a [...] x / i-za-az [...]-na?-ri szu-ma# [...]-x-a isz-tu [...] a-na _iti-kam_ [...]-la2-ak [...]-lik _dumu_ u2-[...] [...] x x [...]',
   '... ... ... ... ... ... ... ... ... whose name you wrote ... ... ... he shall stand ... ... ... ... from ... to the month of Adar ...')],
 'P361419': [('_kiszib3_ bu-ka3-nim _dumu_ szu-su2-en6 sza 1/3(disz) _ma-na ku3-babbar_ s,a-ru-pa2-am szi2-im _geme2_-tim sza _e2_ na-ap2-li-is ki-ma _dumu-munus_ na-ap2-li-is gu5-ba-ab2-tim i-di2-a-szur3 _dumu_ pa2-pa2-lim i-na ka

In [12]:
with open('AICC_Failed_Transliterations_List.json', 'w', encoding='utf-8') as f:
    json.dump(failed_p_ids, f, indent=4, ensure_ascii=False)

In [11]:
len(list(p_id_to_tuples.keys())) + len(failed_p_ids)

7149

In [1]:
import json

In [5]:
with open('AICC_Extracted_Transliterations_And_Translations.json', 'r') as f:
    p_id_to_tuples = json.load(f)

In [6]:
p_id_to_tuples

{'P361718': [['1(disz) 5/6(disz) _ma-na_ 1(disz) 5/6(disz) _gin2_',
   '1 5/6 mina 1 5/6 shekels'],
  ['isz-pu-ru-nim-ma a-na ta-aq-na-bi4-im u3 ma-nu-ki-a-szur i-na ka3-ni-isz ta#-di2-in _igi#_ a-szur-i-mi3-ti2 _dumu_ i-di2-we-er',
   "sent, and to Taqnabum and Manuki-Assur in the ka'nish he gave. Before Ashur-imitti, son of Idi-wer."]],
 'P358927': [['[...] x x x [...] [...]-ri#?-a li-x-[...] [...] x su2-en6 [...] sza# ta-asz2-pu-ra#-[ni] [...]-x-ti2-ka3 / szu-mi a-[x] [...] i-ku-pi3-a [...] x / i-za-az [...]-na?-ri szu-ma# [...]-x-a isz-tu [...] a-na _iti-kam_ [...]-la2-ak [...]-lik _dumu_ u2-[...] [...] x x [...]',
   '... ... ... ... ... ... ... ... ... whose name you wrote ... ... ... he shall stand ... ... ... ... from ... to the month of Adar ...']],
 'P361419': [['_kiszib3_ bu-ka3-nim _dumu_ szu-su2-en6 sza 1/3(disz) _ma-na ku3-babbar_ s,a-ru-pa2-am szi2-im _geme2_-tim sza _e2_ na-ap2-li-is ki-ma _dumu-munus_ na-ap2-li-is gu5-ba-ab2-tim i-di2-a-szur3 _dumu_ pa2-pa2-lim i-na ka

In [7]:
len(p_id_to_tuples.keys())

4419

In [11]:
df_train = pd.read_csv('train.csv')
df_PT = pd.read_csv('published_texts.csv')

In [14]:
train_o_id = list(set(df_train['oare_id']))
valid_df_PT = df_PT[df_PT['oare_id'].apply(lambda x : x in train_o_id)].reset_index(drop = True)

In [20]:
cdli_id_to_o_id = {cdli_id:o_id for o_id,cdli_id in zip(valid_df_PT['oare_id'], valid_df_PT['cdli_id'])}

In [22]:
len(cdli_id_to_o_id.keys())

1493

In [27]:
len(list(train_pairs.keys()))

20